In [18]:
# # Let's bring the urls with client id and client secret
# authorization_url = "http://localhost:8080/api/method/frappe.integrations.oauth2.authorize"
# token_refresh_url = "http://localhost:8080/api/method/frappe.integrations.oauth2.get_token"

# client_id = "j6o9atqc5o"
# client_secret = "86c3b15aba"

# Let's bring the urls with client id and client secret
authorization_url = "https://erpnext-emg-opp.m.frappe.cloud/api/method/frappe.integrations.oauth2.authorize"
token_refresh_url = "https://erpnext-emg-opp.m.frappe.cloud/api/method/frappe.integrations.oauth2.get_token"

client_id = "dqgvv7q0bn"
client_secret = "37179dc9e4"

In [35]:
import os
import json
import time
import logging
import threading
import requests
from typing import Dict, Any

logger = logging.getLogger("agent_system.auth")

class ERPNextAuthorizationCodeManager:
    """
    Manages OAuth2 Authorization Code flow tokens for ERPNext.
    Uses long-lived Refresh Tokens to fetch fresh Access Tokens completely in the background,
    avoiding interactive browser permission popups after initial setup.
    """
    def __init__(self, token_url: str, client_id: str, client_secret: str, scope: str = "all openid", cache_file: str = "tokens2.json"):
        self.token_url = token_url
        self.client_id = client_id
        self.client_secret = client_secret
        self.scope = scope
        self.cache_file = cache_file
        
        self._lock = threading.Lock()
        
        # Core runtime credentials state
        self._access_token: str | None = None
        self._refresh_token: str | None = None
        self._expires_at: float = 0.0
        self.expiry_buffer = 300  # 5-minute proactive refresh window

        # Load tokens from storage if they exist from a previous run
        self._load_token_cache()

    def _load_token_cache(self) -> str:
        """Loads existing tokens from disk to maintain state across process restarts."""
        if os.path.exists(self.cache_file):
            try:
                with open(self.cache_file, "r") as f:
                    data = json.load(f)
                    self._access_token = data.get("access_token")
                    self._refresh_token = data.get("refresh_token")
                    self._expires_at = float(data.get("expires_at", 0.0))
                print(data, "data found")
                logger.info("Loaded active session state from disk cache.")
            except Exception as e:
                logger.warning(f"Could not read token cache file: {e}. Starting fresh.")

    def _save_token_cache(self) :
        """Saves active tokens securely to disk storage."""
        print("start")
        try:
            with open(self.cache_file, "w") as f:
                json.dump({
                    "access_token": self._access_token,
                    "refresh_token": self._refresh_token,
                    "expires_at": self._expires_at
                }, f)
        except Exception as e:
            logger.error(f"Failed to persist tokens to disk cache: {e}")

    def _refresh_access_token(self) -> None:
        """Uses the long-lived refresh_token to securely pull a fresh 1-hour access_token."""
        if not self._refresh_token:
            raise ValueError(
                "No Refresh Token found! You must perform the initial interactive browser authorization "
                "via Postman or a configuration endpoint once, then populate the tokens.json cache file."
            )

        # Preparing fields based on your configuration parameters
        payload = {
            "grant_type": "refresh_token",
            "refresh_token": self._refresh_token,
            "client_id": self.client_id,
            "client_secret": self.client_secret,
            "scope": self.scope
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}

        logger.info("Proactively exchanging refresh token for a new access token...")
        try:
            response = requests.post(self.token_url, data=payload, headers=headers, timeout=10)
            response.raise_for_status()
            data = response.json()

            self._access_token = data["access_token"]
            # Update refresh token if the server rolls/rotates it
            if "refresh_token" in data:
                self._refresh_token = data["refresh_token"]

            expires_in = float(data.get("expires_in", 3600))
            self._expires_at = time.time() + expires_in

            # Sync to storage immediately
            self._save_token_cache()
            logger.info("Tokens successfully rotated and stored securely.")
            
        except requests.RequestException as e:
            logger.critical(f"Failed to refresh background access token: {e}")
            raise RuntimeError(f"ERPNext rejected token rotation payload: {e}") from e

    def get_valid_token(self) -> str:
        """Thread-safe gateway ensuring agents always receive an active access token."""
        current_time = time.time()
        
        if current_time >= (self._expires_at - self.expiry_buffer):
            with self._lock:
                if time.time() >= (self._expires_at - self.expiry_buffer):
                    self._refresh_access_token()
                    
        return self._access_token


In [36]:
test = ERPNextAuthorizationCodeManager(client_id=client_id, client_secret=client_secret, token_url=token_refresh_url)

In [38]:
print(test._load_token_cache())

None


In [8]:
from requests.auth import AuthBase

class ERPNextBearerAuth(AuthBase) :

    def __init__(self, token_manager) :
        self.token_manager = token_manager

    def __call__(self, r) :

        print("Accepting the valid token and creating authorization payload")
        token = self.token_manager.get_valid_token()
        r.headers["Authorization"] = f"Bearer {token}"

        print(f"r--- value -- {r.headers["Authorization"]}")

        return r


In [9]:
import requests
from typing import Dict, Any

class ERPNextBaseClient:
    """
    Base API Client wrapping connections to ERPNext with auto-refreshing sessions.
    """
    def __init__(self, base_url: str, token_manager: ERPNextAuthorizationCodeManager):
        self.base_url = base_url.rstrip('/')
        self.session = requests.Session()
        
        # Seamless integration: Attach the auto-refreshing authorization engine
        self.session.auth = ERPNextBearerAuth(token_manager)
        
        # Enforce standard headers for ERPNext REST architecture
        self.session.headers.update({
            "Accept": "application/json",
            "Content-Type": "application/json"
        })

    def request(self, method: str, endpoint: str, **kwargs) -> Dict[str, Any]:
        """
        Centralized request method with forced production timeouts and status handling.
        """
        url = f"{self.base_url}/{endpoint.lstrip('/')}"
        
        # Default production protection constraints
        if "timeout" not in kwargs:
            kwargs["timeout"] = 30  # Keep agent threads from hanging indefinitely
            
        try:
            response = self.session.request(method, url, **kwargs)
            response.raise_for_status()
            
            # ERPNext wraps standard REST data inside a 'data' key object
            return response.json()
        except requests.HTTPError as e:
            if response.status_code == 401:
                # This should rarely occur due to our proactive token buffer checks
                raise PermissionError("ERPNext rejected credentials with 401 Unauthorized.") from e
            raise RuntimeError(f"ERPNext API returned unexpected error {response.status_code}: {response.text}") from e


In [10]:
from pathlib import Path
from pydantic import Field, HttpUrl
from pydantic_settings import BaseSettings, SettingsConfigDict

# Automatically locate the directory containing config/settings.py, then go up 1 level to the project root
PROJECT_ROOT = Path.cwd().resolve().parent.parent

class AppSettings(BaseSettings):
    """
    Validates and manages all application configuration parameters.
    Automatically loads variables from environment or a local .env file.
    """
    # ---------------------------------------------------------
    # 1. Core ERPNext Configurations (Strict Validation)
    # ---------------------------------------------------------
    erpnext_base_url: HttpUrl = Field(..., alias="ERPNEXT_BASE_URL")
    token_url : HttpUrl = Field(..., alias="TOKEN_REFRESH_URL")
    erpnext_client_id: str = Field(..., alias="ERPNEXT_CLIENT_ID")
    erpnext_client_secret: str = Field(..., alias="ERPNEXT_CLIENT_SECRET")
    erpnext_scope: str = Field(default="all openid", alias="ERPNEXT_SCOPE")
    
    # ---------------------------------------------------------
    # 2. Application & Operations Settings
    # ---------------------------------------------------------
    app_env: str = Field(default="production", alias="APP_ENV")
    log_level: str = Field(default="INFO", alias="LOG_LEVEL")


    @property
    def absolute_token_path(self) -> str:
        """Forces tokens.json to always resolve to the absolute base project root directory."""
        return str(PROJECT_ROOT / "tokens.json")

    # ---------------------------------------------------------
    # 4. Configuration Metadata Layer
    # ---------------------------------------------------------
    # Configures Pydantic to read directly from a local `.env` file
    model_config = SettingsConfigDict(
        env_file=".env", 
        env_file_encoding="utf-8", 
        extra="ignore"  # Safely ignores unexpected system environment variables
    )

# 5. Instantiate a single, global settings instance to import anywhere
settings = AppSettings()


In [11]:
import os
import sys
import time

def run_integration_wiring_test():
    print("=============================logger.info================")
    print("🚀 STARTING ERPNEXT AUTH MODULE WIRING TEST")
    print("==================================================")

    # 1. Component Verification: Load Pydantic Config settings
    print(f"[1] Environment settings initialized successfully.")
    print(f"    - Target Server: {settings.erpnext_base_url}")
    print(f"    - Cache Target:  {settings.token_url}")
    print(f"    - Cache Target:  {settings.absolute_token_path}")

    # 2. Component Verification: Instantiate the Token Manager
    print("\n[2] Wiring up the Token Manager Engine...")
    token_manager = ERPNextAuthorizationCodeManager(
        token_url=settings.token_url,
        client_id=settings.erpnext_client_id,
        client_secret=settings.erpnext_client_secret,
        cache_file=settings.absolute_token_path
    )

    print("[3] generating the token")
    token = token_manager.get_valid_token()

    # 3. Component Verification: Wire the Manager into the HTTP Base Client
    print("[3] Wiring Token Manager into the ERPNext Base Client...")
    client = ERPNextBaseClient(
        base_url=str(settings.erpnext_base_url), 
        token_manager=token_manager
    )

    payload = {
        "material_request_type": "Purchase",
        "schedule_date": "2026-12-01",
        "company": "Maahi's Foudation",
        "items": [
            {
            "item_code": "ITEM-001",
            "item_name": "MacBook Pro M3",          
            "description": "16-inch, 32GB RAM, 1TB SSD - Space Black", 
            "qty": 2,                               
            "rate": 250000,                         
            "amount": 500000,                       
            "brand": "Apple",                       
            "item_group": "IT Hardware",            
            "schedule_date": "2026-12-01",
            "warehouse": "Stores - MF"
            }
        ]
    }
    

    # 4. Component Verification: Execute Live Connection and Exception Catching Loop
    print("\n[4] Triggering Live Integrated Auth Verification Round...")
    try:
        # We test by forcing a standard read request to check permissions
        # This triggers: Client -> Interceptor -> Token Manager Cache Check -> Network Sync
        print("    -> Sending request to read active Material Requests...")
        # response_data = client.request("GET", "api/resource/Purchase Order")
        response_data = client.request("POST", "api/resource/Material Request", json=payload)
        
        print("\n==================================================")
        print("🎉 SUCCESS: All components are wired together perfectly!")
        print("==================================================")
        print(f"Server Sample Response: {response_data}")
        
    except Exception as e:
        print("     -> Error while making request", e)


if __name__ == "__main__":
    print("🔥 containers.py execution started")
    run_integration_wiring_test()



🔥 containers.py execution started
=============================logger.info================
🚀 STARTING ERPNEXT AUTH MODULE WIRING TEST
[1] Environment settings initialized successfully.
    - Target Server: http://localhost:8080/
    - Cache Target:  http://localhost:8080/api/method/frappe.integrations.oauth2.get_token
    - Cache Target:  D:\AI-Restart-2026\Industry Level Agentic AI Projects - 26-July-2026\PR-TO-PO-Complete-AgenticAI-Systen\tokens.json

[2] Wiring up the Token Manager Engine...
[3] generating the token
[3] Wiring Token Manager into the ERPNext Base Client...

[4] Triggering Live Integrated Auth Verification Round...
    -> Sending request to read active Material Requests...
Accepting the valid token and creating authorization payload
r--- value -- Bearer WksCA478nBUbClPzgaFYu0qMDRklMo

🎉 SUCCESS: All components are wired together perfectly!
Server Sample Response: {'data': {'name': 'MAT-MR-2026-00048', 'owner': 'mahendragharad5@gmail.com', 'creation': '2026-10-01 00:02

In [7]:
payload = {
    "message": {
        "name": "MAT-MR-2026-00020",
        "owner": "mahendragharad5@gmail.com",
        "creation": "2026-09-13 11:37:18.405640",
        "modified": "2026-09-13 11:37:18.405640",
        "modified_by": "mahendragharad5@gmail.com",
        "docstatus": 0,
        "idx": 0,
        "naming_series": "MAT-MR-.YYYY.-",
        "title": "Quarterly IT & Hardware Procurement",
        "material_request_type": "Purchase",
        "customer": "",
        "company": "Maahi's Foudation",
        "auto_created_via_reorder": 0,
        "transaction_date": "2026-09-10",
        "schedule_date": "2026-09-25",
        "buying_price_list": "Standard Buying",
        "amended_from": 1,
        "scan_barcode": 1,
        "last_scanned_warehouse": 1,
        "set_from_warehouse": 1,
        "set_warehouse": "All Warehouses - MF",
        "tc_name": 1,
        "terms": 1,
        "status": "Draft",
        "per_ordered": 0,
        "transfer_status": "",
        "per_received": 0,
        "letter_head": "Company Letterhead - Grey",
        "select_print_heading": 1,
        "job_card": 1,
        "work_order": 1,
        "doctype": "Material Request",
        "items": [
            {
                "name": "pv06ivvo5i",
                "owner": "mahendragharad5@gmail.com",
                "creation": "2026-09-13 11:37:18.405640",
                "modified": "2026-09-13 11:37:18.668103",
                "modified_by": "mahendragharad5@gmail.com",
                "docstatus": 0,
                "idx": 1,
                "item_code": "ITEM-001",
                "item_name": "Test Item",
                "schedule_date": "2026-09-25",
                "description": "High-performance workstation component with extended 3-year hardware warranty.",
                "item_group": "Consumable",
                "brand": 1,
                "image": "",
                "qty": 15,
                "stock_uom": "Nos",
                "from_warehouse": 1,
                "warehouse": "All Warehouses - MF",
                "uom": "Nos",
                "conversion_factor": 1,
                "stock_qty": 15,
                "min_order_qty": 0,
                "projected_qty": 15,
                "picked_qty": 0,
                "actual_qty": 0,
                "ordered_qty": 0,
                "received_qty": 0,
                "rate": 100,
                "price_list_rate": 0,
                "amount": 1500,
                "expense_account": "Cost of Goods Sold - MF",
                "wip_composite_asset": 1,
                "manufacturer": 1,
                "manufacturer_part_no": 1,
                "bom_no": 1,
                "project": 1,
                "cost_center": "Main - MF",
                "lead_time_date": 1,
                "sales_order": 1,
                "sales_order_item": 1,
                "packed_item": 1,
                "production_plan": 1,
                "material_request_plan_item": 1,
                "job_card_item": 1,
                "projected_on_hand": 0,
                "reorder_level": 0,
                "reorder_qty": 0,
                "page_break": 0,
                "parent": "MAT-MR-2026-00020",
                "parentfield": "items",
                "parenttype": "Material Request",
                "doctype": "Material Request Item",
                "__unsaved": 1
            }
        ]
    },
    "_server_messages": "[\"{\\\"message\\\":\\\"Item rates have been updated based on the selected Buying Price List Standard Buying\\\",\\\"as_table\\\":false,\\\"title\\\":\\\"Message\\\",\\\"alert\\\":1}\"]"
}

In [8]:
payload.get("message")

{'name': 'MAT-MR-2026-00020',
 'owner': 'mahendragharad5@gmail.com',
 'creation': '2026-09-13 11:37:18.405640',
 'modified': '2026-09-13 11:37:18.405640',
 'modified_by': 'mahendragharad5@gmail.com',
 'docstatus': 0,
 'idx': 0,
 'naming_series': 'MAT-MR-.YYYY.-',
 'title': 'Quarterly IT & Hardware Procurement',
 'material_request_type': 'Purchase',
 'customer': '',
 'company': "Maahi's Foudation",
 'auto_created_via_reorder': 0,
 'transaction_date': '2026-09-10',
 'schedule_date': '2026-09-25',
 'buying_price_list': 'Standard Buying',
 'amended_from': 1,
 'scan_barcode': 1,
 'last_scanned_warehouse': 1,
 'set_from_warehouse': 1,
 'set_warehouse': 'All Warehouses - MF',
 'tc_name': 1,
 'terms': 1,
 'status': 'Draft',
 'per_ordered': 0,
 'transfer_status': '',
 'per_received': 0,
 'letter_head': 'Company Letterhead - Grey',
 'select_print_heading': 1,
 'job_card': 1,
 'work_order': 1,
 'doctype': 'Material Request',
 'items': [{'name': 'pv06ivvo5i',
   'owner': 'mahendragharad5@gmail.com

In [9]:
response = payload.get("message",{})
response.get("items", {})

[{'name': 'pv06ivvo5i',
  'owner': 'mahendragharad5@gmail.com',
  'creation': '2026-09-13 11:37:18.405640',
  'modified': '2026-09-13 11:37:18.668103',
  'modified_by': 'mahendragharad5@gmail.com',
  'docstatus': 0,
  'idx': 1,
  'item_code': 'ITEM-001',
  'item_name': 'Test Item',
  'schedule_date': '2026-09-25',
  'description': 'High-performance workstation component with extended 3-year hardware warranty.',
  'item_group': 'Consumable',
  'brand': 1,
  'image': '',
  'qty': 15,
  'stock_uom': 'Nos',
  'from_warehouse': 1,
  'warehouse': 'All Warehouses - MF',
  'uom': 'Nos',
  'conversion_factor': 1,
  'stock_qty': 15,
  'min_order_qty': 0,
  'projected_qty': 15,
  'picked_qty': 0,
  'actual_qty': 0,
  'ordered_qty': 0,
  'received_qty': 0,
  'rate': 100,
  'price_list_rate': 0,
  'amount': 1500,
  'expense_account': 'Cost of Goods Sold - MF',
  'wip_composite_asset': 1,
  'manufacturer': 1,
  'manufacturer_part_no': 1,
  'bom_no': 1,
  'project': 1,
  'cost_center': 'Main - MF',
 

In [10]:
{
  "naming_series": "PUR-ORD-.YYYY.-",
  "transaction_date": "2026-09-10",
  "schedule_date": "2026-09-25",
  "company": "Maahi's Foudation",
  "supplier": "Test Supplier API", 
  "currency": "INR",
  "buying_price_list": "Standard Buying",
  "set_warehouse": "All Warehouses - MF",
  "items": [
    {
      "item_code": "ITEM-001",
      "qty": 15,
      "uom": "Nos",
      "rate": 100,
      "warehouse": "All Warehouses - MF",
      "material_request": "MAT-MR-2026-00015",
      "material_request_item": "8lsujejrcg"
    }
  ]
}


{'naming_series': 'PUR-ORD-.YYYY.-',
 'transaction_date': '2026-09-10',
 'schedule_date': '2026-09-25',
 'company': "Maahi's Foudation",
 'supplier': 'Test Supplier API',
 'currency': 'INR',
 'buying_price_list': 'Standard Buying',
 'set_warehouse': 'All Warehouses - MF',
 'items': [{'item_code': 'ITEM-001',
   'qty': 15,
   'uom': 'Nos',
   'rate': 100,
   'warehouse': 'All Warehouses - MF',
   'material_request': 'MAT-MR-2026-00015',
   'material_request_item': '8lsujejrcg'}]}

In [1]:
import json
import re
import requests

def parse_erpnext_error(response_json: dict | str) -> dict:
    """
    Parses complex nested ERPNext/Frappe exception payloads, unescaping 
    `_server_messages` into clean error summaries suitable for LLM context.
    """
    if isinstance(response_json, str):
        try:
            response_json = json.loads(response_json)
        except json.JSONDecodeError:
            return {"message": response_json, "exc_type": "UnknownError"}

    exc_type = response_json.get("exc_type", "ERPNextError")
    server_messages_raw = response_json.get("_server_messages")
    
    clean_messages = []

    if server_messages_raw:
        try:
            # Step 1: Handle stringified JSON inside _server_messages
            messages_list = (
                json.loads(server_messages_raw) 
                if isinstance(server_messages_raw, str) 
                else server_messages_raw
                
            )
            
            for msg_item in messages_list:
                # Step 2: Unescape nested JSON object if wrapped as string
                if isinstance(msg_item, str):
                    try:
                        msg_item = json.loads(msg_item)
                    except json.JSONDecodeError:
                        pass
                
                # Step 3: Extract main error message text
                if isinstance(msg_item, dict) and "message" in msg_item:
                    msg_text = msg_item["message"]
                else:
                    msg_text = str(msg_item)

                # Step 4: Strip HTML tags (Frappe wraps some messages in HTML)
                clean_text = re.sub(r'<[^>]+>', '', msg_text).strip()
                if clean_text:
                    clean_messages.append(clean_text)

        except (json.JSONDecodeError, TypeError):
            clean_messages.append(str(server_messages_raw))

    # Fallback to general message or exception string if _server_messages was empty
    if not clean_messages:
        main_msg = response_json.get("message") or response_json.get("exception") or "An unknown error occurred."
        clean_messages.append(re.sub(r'<[^>]+>', '', str(main_msg)).strip())

    return {
        "exc_type": exc_type,
        "message": " | ".join(clean_messages)
    }

In [2]:
import json
import re
from typing import Any


def _strip_html(text: str) -> str:
    """Remove HTML tags and normalize whitespace."""
    if not text:
        return ""
    text = re.sub(r"<[^>]+>", "", text)
    return " ".join(text.split())


def parse_erpnext_error(response_json: dict | str) -> dict:
    """
    Parse nested ERPNext/Frappe exception payloads and return a clean,
    LLM-friendly error summary.

    Returns:
        {
            "exc_type": str,
            "message": str,          # single-line, joined error summary
            "details": list[str]     # individual cleaned messages (optional, useful for agents)
        }
    """
    # Normalize input
    if isinstance(response_json, str):
        try:
            response_json = json.loads(response_json)
        except json.JSONDecodeError:
            return {
                "exc_type": "UnknownError",
                "message": _strip_html(response_json),
                "details": [_strip_html(response_json)],
            }

    if not isinstance(response_json, dict):
        return {
            "exc_type": "UnknownError",
            "message": "An unknown error occurred.",
            "details": ["An unknown error occurred."],
        }

    exc_type = response_json.get("exc_type") or "ERPNextError"
    server_messages_raw = response_json.get("_server_messages")

    clean_messages: list[str] = []

    if server_messages_raw:
        # Normalize to a list
        try:
            messages_list = (
                json.loads(server_messages_raw)
                if isinstance(server_messages_raw, str)
                else server_messages_raw
            )
        except (json.JSONDecodeError, TypeError):
            messages_list = [server_messages_raw]

        if isinstance(messages_list, dict):
            # Sometimes it's a single object instead of a list
            messages_list = [messages_list]

        if isinstance(messages_list, list):
            for msg_item in messages_list:
                # Unescape nested JSON string if present
                if isinstance(msg_item, str):
                    try:
                        msg_item = json.loads(msg_item)
                    except json.JSONDecodeError:
                        pass

                if isinstance(msg_item, dict):
                    msg_text = msg_item.get("message") or msg_item.get("title") or ""
                else:
                    msg_text = str(msg_item) if msg_item is not None else ""

                cleaned = _strip_html(msg_text)
                if cleaned:
                    clean_messages.append(cleaned)

    # Fallback if _server_messages gave us nothing useful
    if not clean_messages:
        main_msg = (
            response_json.get("message")
            or response_json.get("exception")
            or response_json.get("exc")
            or "An unknown error occurred."
        )
        cleaned = _strip_html(str(main_msg))
        if cleaned:
            clean_messages.append(cleaned)
        else:
            clean_messages = ["An unknown error occurred."]

    # Deduplicate while preserving order
    seen = set()
    unique_messages = []
    for m in clean_messages:
        if m not in seen:
            seen.add(m)
            unique_messages.append(m)

    message_str = " | ".join(unique_messages)

    return {
        "exc_type": exc_type,
        "message": message_str,
        "details": unique_messages,
    }

In [3]:
error = {
    "message": "ERPNext server failed to submit Material Request: ERPNext API returned unexpected error 404: {\"exc_type\":\"DoesNotExistError\",\"_server_messages\":\"[\\\"{\\\\\\\"message\\\\\\\":\\\\\\\"Material Request MT-MR-2026-00035 not found\\\\\\\",\\\\\\\"as_table\\\\\\\":false,\\\\\\\"title\\\\\\\":\\\\\\\"Message\\\\\\\",\\\\\\\"indicator\\\\\\\":\\\\\\\"red\\\\\\\",\\\\\\\"raise_exception\\\\\\\":1,\\\\\\\"__frappe_exc_id\\\\\\\":\\\\\\\"6e63e75c83ac5bd51abb6353b31c39fca3bc24c561205eb9733a27f4\\\\\\\"}\\\"]\"}",
    "can_retry": "false",
    "suggestion": "Check missing parameters or correct data properties based on details."
  }

parse_erpnext_error(error)

{'exc_type': 'ERPNextError',
 'message': 'ERPNext server failed to submit Material Request: ERPNext API returned unexpected error 404: {"exc_type":"DoesNotExistError","_server_messages":"[\\"{\\\\\\"message\\\\\\":\\\\\\"Material Request MT-MR-2026-00035 not found\\\\\\",\\\\\\"as_table\\\\\\":false,\\\\\\"title\\\\\\":\\\\\\"Message\\\\\\",\\\\\\"indicator\\\\\\":\\\\\\"red\\\\\\",\\\\\\"raise_exception\\\\\\":1,\\\\\\"__frappe_exc_id\\\\\\":\\\\\\"6e63e75c83ac5bd51abb6353b31c39fca3bc24c561205eb9733a27f4\\\\\\"}\\"]"}',
 'details': ['ERPNext server failed to submit Material Request: ERPNext API returned unexpected error 404: {"exc_type":"DoesNotExistError","_server_messages":"[\\"{\\\\\\"message\\\\\\":\\\\\\"Material Request MT-MR-2026-00035 not found\\\\\\",\\\\\\"as_table\\\\\\":false,\\\\\\"title\\\\\\":\\\\\\"Message\\\\\\",\\\\\\"indicator\\\\\\":\\\\\\"red\\\\\\",\\\\\\"raise_exception\\\\\\":1,\\\\\\"__frappe_exc_id\\\\\\":\\\\\\"6e63e75c83ac5bd51abb6353b31c39fca3bc24c561205eb

In [4]:
from pathlib import Path
from pydantic import Field, HttpUrl
from pydantic_settings import BaseSettings, SettingsConfigDict

# Automatically locate the directory containing config/settings.py, then go up 1 level to the project root
PROJECT_ROOT = Path.cwd().resolve().parent.parent

class AppSettings(BaseSettings):
    """
    Validates and manages all application configuration parameters.
    Automatically loads variables from environment or a local .env file.
    """
    # ---------------------------------------------------------
    # 1. Core ERPNext Configurations (Strict Validation)
    # ---------------------------------------------------------
    erpnext_base_url: HttpUrl = Field(..., alias="ERPNEXT_BASE_URL")
    token_url : HttpUrl = Field(..., alias="TOKEN_REFRESH_URL")
    erpnext_client_id: str = Field(..., alias="ERPNEXT_CLIENT_ID")
    erpnext_client_secret: str = Field(..., alias="ERPNEXT_CLIENT_SECRET")
    erpnext_scope: str = Field(default="all openid", alias="ERPNEXT_SCOPE")
    
    # ---------------------------------------------------------
    # 2. Application & Operations Settings
    # ---------------------------------------------------------
    app_env: str = Field(default="production", alias="APP_ENV")
    log_level: str = Field(default="INFO", alias="LOG_LEVEL")

    @property
    def absolute_token_path(self) -> str:
        """Forces tokens.json to always resolve to the absolute base project root directory."""
        return str(PROJECT_ROOT / "tokens.json")
    
    # ---------------------------------------------------------
    # 3. API Services Vallidations 
    # ---------------------------------------------------------
    groq_api_key : str = Field(..., alias="GROQ_API_KEY")


    # ---------------------------------------------------------
    # 4. Configuration Metadata Layer
    # ---------------------------------------------------------
    # Configures Pydantic to read directly from a local `.env` file
    model_config = SettingsConfigDict(
        env_file=".env", 
        env_file_encoding="utf-8", 
        extra="ignore"  # Safely ignores unexpected system environment variables
    )

# 5. Instantiate a single, global settings instance to import anywhere
settings = AppSettings()


In [5]:
from langchain_groq import ChatGroq


def main():

    # Initialize Groq LLM through LangChain
    llm = ChatGroq(
        model="openai/gpt-oss-20b",
        api_key="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",
        temperature=0,
    )

    # Send a simple message
    response = llm.invoke(
        "Explain what an ERPNext Purchase Requisition is in 3 sentences."
    )

    print("\n========== GROQ RESPONSE ==========\n")
    print(response.content)

    print("\n========== RESPONSE TYPE ==========\n")
    print(type(response))


if __name__ == "__main__":
    main()


========== GROQ RESPONSE ==========

A Purchase Requisition in ERPNext is an internal request created by a department or employee to procure goods or services, capturing details such as item, quantity, and desired delivery date. It serves as a formal approval workflow, allowing managers to review, modify, or approve the request before it becomes a purchase order. Once approved, the requisition can be automatically converted into a purchase order, ensuring that procurement follows company policy and budget constraints.

========== RESPONSE TYPE ==========

<class 'langchain_core.messages.ai.AIMessage'>


In [1]:
from typing import List, Optional, Any
from pydantic import BaseModel, Field, model_validator
from datetime import date
from pydantic import BaseModel, Field, field_validator, model_validator

def _validate_future_date_string(value: Optional[str], field_name: str) -> Optional[str]:
    if value is None:
        return value
    try:
        parsed_date = date.fromisoformat(value)
    except ValueError:
        raise ValueError(f"Invalid {field_name} format. Expected YYYY-MM-DD.")
    
    # Using the current active global timeline 2026
    today = date.today()
    if parsed_date <= today:
        raise ValueError(f"{field_name} '{value}' must be a strictly future date after {today.isoformat()}.")
    return value

class MaterialRequestItemSchema(BaseModel):
    item_code: str = Field(..., description="Item Code in ERPNext")
    qty: float = Field(..., description="Quantity requested")
    uom: str = Field(default="Nos", description="Unit of Measure")
    schedule_date: str = Field(None, description="Delivery date (YYYY-MM-DD)")
    warehouse: str = Field(None, description="Target warehouse")
    rate: float = Field(default=0.0, description="Estimated rate per unit")

    @field_validator("schedule_date")
    @classmethod
    def validate_item_schedule_date(cls, value: Optional[str]) -> Optional[str]:
        return _validate_future_date_string(value, "schedule_date")

class MaterialRequestDocSchema(BaseModel):
    doctype: str = Field(default="Material Request", description="Target DocType")
    material_request_type: str = Field(default="Purchase", description="Type of requisition")
    transaction_date: str = Field(..., description="Posting date (YYYY-MM-DD)")
    schedule_date: str = Field(..., description="Required delivery date (YYYY-MM-DD)")
    company: str = Field(..., description="Company name")
    department: str = Field(..., description="Department name")
    set_warehouse: str = Field(default="All Warehouses - MF", description="Default warehouse")
    title: str = Field(..., description="Requisition title")
    remark : str = Field(..., description="Remarks or justification")
    items: List[MaterialRequestItemSchema] = Field(..., description="List of items to request")

    @field_validator("material_request_type")
    @classmethod
    def validate_request_type(cls, value: str) -> str:
        allowed_types = {"Purchase", "Material Transfer", "Material Issue", "Manufacture", "Customer Provided"}
        if value not in allowed_types:
            raise ValueError(f"Invalid material_request_type. Must be one of: {allowed_types}")
        return value

    @field_validator("items")
    @classmethod
    def validate_items_list(cls, items: List[MaterialRequestItemSchema]) -> List[MaterialRequestItemSchema]:
        if not items:
            raise ValueError("The child table transaction list array cannot be left empty.")
        return items

    @field_validator("transaction_date", "schedule_date")
    @classmethod
    def validate_strict_future_date(cls, value: str) -> str:
        res = _validate_future_date_string(value, "date")
        assert res is not None
        return res

    @model_validator(mode="after")
    def validate_schedule_after_transaction(self) -> "MaterialRequestDocSchema":
        """Ensures schedule_date is not earlier than transaction_date."""
        trans_d = date.fromisoformat(self.transaction_date)
        sched_d = date.fromisoformat(self.schedule_date)

        if sched_d < trans_d:
            raise ValueError(
                f"schedule_date ({self.schedule_date}) cannot be earlier than transaction_date ({self.transaction_date})."
            )
        return self

MaterialRequestDocSchema.model_rebuild()

class ConvertPRToPOInput(BaseModel):
    material_request_id: str = Field(..., description="The unique ID of the Material Request (PR)")
    supplier: str = Field(..., description="The registered Name of the Supplier/Vendor")
    currency: str = Field(default="INR", description="Currency code")

ConvertPRToPOInput.model_rebuild()

class Purchase_Order_Id_Validator(BaseModel):
    purchase_order_id : str = Field(..., description="The purchase order id to submit the final version of purchase order.")

Purchase_Order_Id_Validator.model_rebuild()

class Material_Request_Id_Validator(BaseModel):
    material_request_id : str = Field(..., description="The purchase order id to submit the final version of purchase order.")

Material_Request_Id_Validator.model_rebuild()


In [2]:
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage
state = {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to generate a material request payload. Ensure all required fields are present. The user gave all needed info.\n\nCheck schema: generate_material_request expects doc with fields: company, department, items, remark, schedule_date, set_warehouse, title, transaction_date. Also material_request_type optional default "Purchase". They gave company: "Maahi\'s Foudation" (typo maybe "Foundation"? Use exactly as given). department: Management. title: Quarterly IT Procurement. transaction_date: 30 September 2026 => 2026-09-30. schedule_date: 9 September 2026 => 2026-09-09. set_warehouse: All Warehouses - MF. remark: Direct script integration test. Items: list with one item: item_code ITEM-001, qty 15, rate 2500, uom Nos, warehouse All Warehouses - MF, schedule_date maybe? In items schema schedule_date optional. We can include schedule_date same as overall? They said keep schedule date as 9 September 2026. So include schedule_date for item.\n\nAll numeric values as numbers. Rate 2500 (float or int). Qty 15.\n\nNow produce function call.', 'tool_calls': [{'id': 'fc_b2f42151-1ac4-4881-acf3-e1d52b8b196a', 'function': {'arguments': '{"doc":{"company":"Maahi\'s Foudation","department":"Management","items":[{"item_code":"ITEM-001","qty":15,"rate":2500,"schedule_date":"2026-09-09","uom":"Nos","warehouse":"All Warehouses - MF"}],"material_request_type":"Purchase","remark":"Direct script integration test","schedule_date":"2026-09-09","set_warehouse":"All Warehouses - MF","title":"Quarterly IT Procurement","transaction_date":"2026-09-30"}}', 'name': 'generate_material_request'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 431, 'prompt_tokens': 1551, 'total_tokens': 1982, 'completion_time': 0.905687674, 'completion_tokens_details': {'reasoning_tokens': 248}, 'prompt_time': 0.17240424, 'prompt_tokens_details': None, 'queue_time': 0.410934385, 'total_time': 1.078091914}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bb691ea66b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0c9dd-0427-7c43-b8ee-5f401be58b8e-0', tool_calls=[{'name': 'generate_material_request', 'args': {'doc': {'company': "Maahi's Foudation", 'department': 'Management', 'items': [{'item_code': 'ITEM-001', 'qty': 15, 'rate': 2500, 'schedule_date': '2026-09-09', 'uom': 'Nos', 'warehouse': 'All Warehouses - MF'}], 'material_request_type': 'Purchase', 'remark': 'Direct script integration test', 'schedule_date': '2026-09-09', 'set_warehouse': 'All Warehouses - MF', 'title': 'Quarterly IT Procurement', 'transaction_date': '2026-09-30'}}, 'id': 'fc_b2f42151-1ac4-4881-acf3-e1d52b8b196a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1551, 'output_tokens': 431, 'total_tokens': 1982, 'output_token_details': {'reasoning': 248}})]}

In [3]:
import re
from typing import Any, List
from pydantic import ValidationError

def extract_clean_pydantic_messages(error_context: Any) -> List[str]:
    """
    Parses ValidationError objects, raw error strings, or generic exceptions 
    into clean, field-level actionable statements for LLM self-healing loops.
    """
    clean_errors = []

    # 1. Direct handling of Pydantic ValidationError instances
    if isinstance(error_context, ValidationError):
        for err in error_context.errors():
            loc_path = ".".join(str(loc) for loc in err.get("loc", []))
            msg = err.get("msg", "").strip()
            
            # Remove redundant 'Value error, ' prefix if present
            msg = re.sub(r'^Value error,\s*', '', msg, flags=re.IGNORECASE)

            if loc_path:
                clean_errors.append(f"Field '{loc_path}': {msg}")
            else:
                clean_errors.append(msg)

        if clean_errors:
            return clean_errors

    # Convert whatever input we got (string, dict, custom exception) to string
    raw_str = str(error_context)

    # 2. String parsing with robust Regex pattern matching
    # Matches Pydantic v2 multiline trace pattern: field_path followed by reason lines
    pattern = re.compile(
        r'^(?P<field>[\w\.]+)\n\s*(?P<reason>.*?)(?=\s*\[type=|\n\w+|\Z)', 
        re.DOTALL | re.MULTILINE
    )

    for match in pattern.finditer(raw_str):
        field = match.group("field").strip()
        reason = match.group("reason").strip()

        # Filter out boilerplate metadata lines
        if "validation errors for" in field or "https://errors.pydantic.dev" in field:
            continue

        if reason:
            # Clean off leading 'Value error,' or similar phrasing
            clean_reason = re.sub(r'^(?:Value error,?\s*|Input should be\s*)', '', reason, flags=re.IGNORECASE)
            # Remove any residual trailing error codes like [type=value_error, ...]
            clean_reason = re.sub(r'\[type=.*\]$', '', clean_reason).strip()
            
            clean_errors.append(f"Field '{field}': {clean_reason}")

    # 3. Fallback for non-standard error text or unexpected formats
    if not clean_errors:
        # Strip out standard URL links and Pydantic headers
        cleaned_fallback = re.sub(r'https?://\S+', '', raw_str)
        cleaned_fallback = re.sub(r'\d+ validation errors? for \w+', '', cleaned_fallback).strip()
        
        if cleaned_fallback:
            clean_errors.append(cleaned_fallback)

    return clean_errors

In [4]:
def extract_tool_text(content) -> str:
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        text_parts = []

        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                text_parts.append(block.get("text", ""))

        return "\n".join(text_parts)

    if isinstance(content, dict):
        return json.dumps(content)

    return str(content)

In [5]:
def parse_tool_response(content) -> dict:
    text = extract_tool_text(content)

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return {
            "status": "ERROR",
            "error": {
                "message": text,
                "can_retry": "false"
            }
        }

    if isinstance(parsed, dict):
        return parsed

    return {
        "status": "ERROR",
        "error": {
            "message": "MCP returned a non-object response.",
            "details": parsed,
            "can_retry": "false"
        }
    }

In [6]:
import json
import re

def clean_erpnext_error(raw_error_input):
    """
    Parses multiply-escaped ERPNext/Frappe server errors into a clean, human-readable dictionary.
    """
    if isinstance(raw_error_input, dict):
        message_str = raw_error_input.get("message", "")
    else:
        message_str = str(raw_error_input)

    # 1. Attempt to extract 'message' from raw Frappe _server_messages array
    try:
        # Match server messages list inside string
        server_msg_match = re.search(r'"_server_messages":\s*"(.*?)"\}', message_str)
        if server_msg_match:
            # Unescape backslashes twice to parse nested JSON
            unescaped_json_str = server_msg_match.group(1).encode('utf-8').decode('unicode_escape')
            unescaped_json_str = unescaped_json_str.replace('\\"', '"')
            
            parsed_messages = json.loads(unescaped_json_str)
            if isinstance(parsed_messages, list) and len(parsed_messages) > 0:
                first_msg = json.loads(parsed_messages[0])
                clean_msg = first_msg.get("message")
                if clean_msg:
                    return {
                        "error_type": first_msg.get("title", "DoesNotExistError"),
                        "clean_message": clean_msg,
                        "raw_summary": f"ERPNext Error: {clean_msg}"
                    }
    except Exception:
        pass  # Fallback to standard parsing if regex/json parsing fails

    # 2. Secondary fallback: Extract standard "DoesNotExistError" style messages
    if "DoesNotExistError" in message_str:
        match = re.search(r'Material Request ([A-Za-z0-9-]+) not found', message_str)
        if match:
            doc_id = match.group(1)
            return {
                "error_type": "DoesNotExistError",
                "clean_message": f"Material Request '{doc_id}' was not found in ERPNext.",
                "raw_summary": f"Document {doc_id} does not exist."
            }

    # 3. Universal Fallback
    return {
        "error_type": "ERPNextServerError",
        "clean_message": message_str.split(":")[-1].strip(),
        "raw_summary": message_str
    }

In [45]:
import json
import logging
from typing import Dict, Any, List
from pydantic import ValidationError
from langgraph.runtime import Runtime
from langchain_core.tools import StructuredTool

from langchain_core.messages import AIMessage, ToolMessage
from langgraph.prebuilt import ToolNode

# Setup specialized production logger
logger = logging.getLogger("ERPNext_SafeExecutionBridge")
logger.setLevel(logging.INFO)

def create_safe_mcp_execution_node(mcp_tools: list):
    """
    Closure Factory creating a production-hardened LangGraph Safe Execution Node.
    """
    # The MCP server expects generate_material_request.doc as a plain dictionary.
    # LangChain normally converts that argument to a Pydantic model first.
    notebook_tools = list(mcp_tools)
    underlying_tool_node = ToolNode(notebook_tools)
    
    async def safe_execution_node(state: dict) -> dict:
        print("\n" + "═"*80)
        print(" 🚀 STARTING SAFE MCP EXECUTION GATEWAY")
        print("═"*80)
        
        # 1. Fetch conversational timeline from state
        messages = state.get("messages", [])
        if not messages:
            print("⚠️ [State Guard] No messages found in context state array. Exiting node.")
            return {"messages": []}
            
        last_message = messages[-1]
        print(f"📥 [Incoming Message Type]: {type(last_message).__name__}")
        
        # Guard Clause: Confirm the orchestrator actually emitted structural tool dependencies
        if not isinstance(last_message, AIMessage) or not last_message.tool_calls:
            print("ℹ️ [Execution Bridge Bypassed]: Last message does not contain executable tool calls.")
            return {"messages": []}
            
        # Standardise structural data layout tracking for single/multi tool arrays
        tool_call = last_message.tool_calls[0] if isinstance(last_message.tool_calls, list) else last_message.tool_calls
        tool_name = tool_call["name"]
        tool_call_id = tool_call["id"]
        
        print(f"🎯 [Target Tool Detected]: {tool_name}")
        print(f"🆔 [Tool Call ID]: {tool_call_id}")
        
        # Extract the dictionary of arguments generated by the orchestrator engine
        original_args = tool_call.get("args", {})
        print(f"📦 [Original Unfiltered Args Passed by LLM]:\n{json.dumps(original_args, indent=2)}")
        
        normalized_args = original_args.copy()

        # Clone and seed retry boundary logs state variables
        retry_counter = state.get("retry_counter", {}).copy()
        if tool_name not in retry_counter:
            retry_counter[tool_name] = 0

        # =====================================================================
        # STEP A: THE DEFENSIVE RUNTIME INTERCEPTOR GATEWAYS
        # =====================================================================
        if "ctx" in normalized_args and isinstance(normalized_args["ctx"], dict):
            print("🧹 [Interceptor Active]: Stripping 'ctx' envelope wrapper from payload.")
            normalized_args = normalized_args["ctx"]
            tool_call["args"] = normalized_args
            print(f"✨ [Post-Ctx Stripped Args]:\n{json.dumps(normalized_args, indent=2)}")
            
        elif "payload" in normalized_args and isinstance(normalized_args["payload"], dict):
            print("🧹 [Interceptor Active]: Stripping 'payload' envelope wrapper from payload.")
            normalized_args = normalized_args["payload"]
            tool_call["args"] = normalized_args
            print(f"✨ [Post-Payload Stripped Args]:\n{json.dumps(normalized_args, indent=2)}")

        # =====================================================================
        # STEP B: PRE-NETWORK LOCAL PYDANTIC CONTRACT VALIDATION
        # =====================================================================
        try:
            print(f"📋 [Validation Stage]: Running schema rules logic for [{tool_name}]...")
            
            if tool_name == "generate_material_request":
                # Extract internal content safely if wrapped in a {"doc": {...}} object
                raw_doc_payload = normalized_args.get("doc", normalized_args)
                print(f"🔍 [Extracted Target Document Payload for Schema Parsing]:\n{json.dumps(raw_doc_payload, indent=2)}")
                
                # Parse against contract definitions
                validated_doc_model = MaterialRequestDocSchema.model_validate(raw_doc_payload)
                print("✨ [Pydantic Contract Instance Generated Successfully]")
                
                # Send a plain dictionary because the MCP server checks for dict explicitly.
                print("BEFORE NORMALIZATION")
                print("args:", tool_call["args"])
                print("args type:", type(tool_call["args"]))

                tool_call["args"] = {
                    "doc": validated_doc_model.model_dump()
                }

                print("AFTER NORMALIZATION")
                print("args:", tool_call["args"])
                print("args type:", type(tool_call["args"]))
                print(f"✅ [Finalized Output Transport Arguments Built]:\n{json.dumps(tool_call['args'], indent=2)}")

            elif tool_name == "submit_material_request":
                print("🔬 Running validation via Material_Request_Id_Validator...")
                Material_Request_Id_Validator.model_validate(normalized_args)

            elif tool_name in ["submit_purchase_order", "cancel_purchase_order", "delete_purchase_order"]:
                print("🔬 Running validation via Purchase_Order_Id_Validator...")
                Purchase_Order_Id_Validator.model_validate(normalized_args)
                
            elif tool_name == "convert_pr_to_draft_po":
                print("🔬 Running validation via ConvertPRToPOInput...")
                ConvertPRToPOInput.model_validate(normalized_args)
                
            else:
                print("🔬 Tool skipped custom pipeline. Falling back to dynamic server definition rules...")
                target_tool = next((t for t in mcp_tools if t.name == tool_name), None)
                if target_tool and hasattr(target_tool, "args_schema") and target_tool.args_schema:
                    target_tool.args_schema.model_validate(normalized_args)

            print(f"🎉 SUCCESS: Validation pipeline completed perfectly for tool: [{tool_name}]. Sending to execution network.")

            # MCP StructuredTools are async-only, so use the async ToolNode path.
            # return await underlying_tool_node.ainvoke(state, runtime=Runtime())

            # 1. Execute underlying node
            base_output = await underlying_tool_node.ainvoke(state, runtime=Runtime())
            emitted_messages = base_output.get("messages", [])

            if not emitted_messages:
                raise RuntimeError(f"MCP tool '{tool_name}' returned no ToolMessage.")

            # 2. Parse response
            last_res = emitted_messages[-1]
            tool_response = parse_tool_response(last_res.content)

            status = str(tool_response.get("status", "")).upper()
            mcp_error = tool_response.get("error")

            logger.debug(f"Parsed MCP Response for [{tool_name}]: {tool_response}")

            # 3. Handle MCP Functional / Application Error
            if status == "ERROR" or mcp_error:
                retry_counter[tool_name] = retry_counter.get(tool_name, 0) + 1

                return {
                    "messages": emitted_messages,
                    "retry_counter": retry_counter,
                    "last_error_context": {
                        "source": "mcp_tool",
                        "tool": tool_name,
                        "error_type": "MCPToolError",
                        "response": tool_response
                    }
                }

            # 4. Success Path
            return {
                "messages": emitted_messages,
                "retry_counter": retry_counter,
                "last_error_context": None
            }

        except ValidationError as pydantic_err:
            logger.error(f"CRITICAL SCHEMA INTERCEPT: Local validation failed for tool [{tool_name}].")
            retry_counter[tool_name] = retry_counter.get(tool_name, 0) + 1

            return {
                "messages": emitted_messages,
                "retry_counter": retry_counter,
                "last_error_context": {
                    "source": "local_validation",
                    "tool": tool_name,
                    "error_type": "ValidationError",
                    "details": pydantic_err.errors()
                }
            }

        except Exception as execution_error:
            logger.error(f"TOOL EXECUTION FAILURE for [{tool_name}]: {execution_error}", exc_info=True)
            retry_counter[tool_name] = retry_counter.get(tool_name, 0) + 1

            return {
                "messages": emitted_messages,
                "retry_counter": retry_counter,
                "last_error_context": {
                    "source": "execution_bridge",
                    "tool": tool_name,
                    "error_type": type(execution_error).__name__,
                    "message": str(execution_error)
                }
            }

    return safe_execution_node

In [11]:
async def run_mcp_http_agent():
    client = MultiServerMCPClient({
        "erpnext": {
            "transport": "streamable_http",          # note: underscore here, adapter expects this form
            "url": "http://127.0.0.1:8000/mcp",       # full URL: host + port + mount_path
        }
    })


    print("⏳ Connecting to Streamable HTTP MCP server at /mcp...")
    tools = await client.get_tools()
    print(f"✅ Successfully loaded {len(tools)} tools: {[t.name for t in tools]}")
    return tools


In [12]:
# Create Material Request
state1 = {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to parse user request.\n\nUser wants to create a Material Request of type Purchase for Maahi\'s Foundation (note spelling "Foudation"? We\'ll keep as provided). Department: Management. transaction_date: 27 November 2026. Must be future relative to current date (2026-09-24). 2026-11-27 is future, okay.\n\nschedule_date (overall request schedule_date): they gave schedule date as 30 December 2026. Must be after transaction date, 2026-12-30 > 2026-11-27, okay.\n\nItem: qty 15 Nos, item_code ITEM-001, rate 2500 per unit, item warehouse All Warehouses - MF, schedule_date for item also 30 December 2026 (same as overall schedule_date). Items list.\n\nAll required fields present.\n\nNow we need to produce generate_material_request payload with doc containing fields.\n\nCheck schema: doc fields: company, department, doctype optional default "Material Request", items list, material_request_type optional default "Purchase", remark, schedule_date, set_warehouse optional default "All Warehouses - MF", title, transaction_date.\n\nWe have company: "Maahi\'s Foudation" (typo). department: "Management". material_request_type: "Purchase". transaction_date: "2026-11-27". schedule_date: "2026-12-30". set_warehouse: "All Warehouses - MF". title: "Quarterly IT Procurement". remark: "Direct script integration test". items: list with one item: item_code "ITEM-001", qty 15, rate 2500, schedule_date "2026-12-30", warehouse "All Warehouses - MF". uom default "Nos" (we can omit). Good.\n\nNow produce tool call.\n\n', 'tool_calls': [{'id': 'fc_c4b38392-397f-492d-b1ea-28444537514b', 'function': {'arguments': '{"doc":{"company":"Maahi\'s Foudation","department":"Management","items":[{"item_code":"ITEM-001","qty":15,"rate":2500,"schedule_date":"2026-12-30","warehouse":"All Warehouses - MF"}],"material_request_type":"Purchase","remark":"Direct script integration test","schedule_date":"2026-12-30","set_warehouse":"All Warehouses - MF","title":"Quarterly IT Procurement","transaction_date":"2026-11-27"}}', 'name': 'generate_material_request'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 551, 'prompt_tokens': 1704, 'total_tokens': 2255, 'completion_time': 1.154652523, 'completion_tokens_details': {'reasoning_tokens': 376}, 'prompt_time': 0.077252081, 'prompt_tokens_details': None, 'queue_time': 0.214096917, 'total_time': 1.231904604}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_86dc843b12', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0d475-fe59-7300-a449-f36fdc7da3f1-0', tool_calls=[{'name': 'generate_material_request', 'args': {'doc': {'company': "Maahi's Foudation", 'department': 'Management', 'items': [{'item_code': 'ITEM-001', 'qty': 15, 'rate': 2500, 'schedule_date': '2026-12-30', 'warehouse': 'All Warehouses - MF'}], 'material_request_type': 'Purchase', 'remark': 'Direct script integration test', 'schedule_date': '2026-12-30', 'set_warehouse': 'All Warehouses - MF', 'title': 'Quarterly IT Procurement', 'transaction_date': '2026-11-27'}}, 'id': 'fc_c4b38392-397f-492d-b1ea-28444537514b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1704, 'output_tokens': 551, 'total_tokens': 2255, 'output_token_details': {'reasoning': 376}})]}

In [13]:
# Submit material request dummy
state2 = {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to process user request: "help me to submit the purchase request with purchase id MT-00001". This is about submitting a material request (purchase request) with ID MT-00001. The appropriate tool is submit_material_request, which takes material_request_id.\n\nWe must check mandatory fields: The tool requires material_request_id. Provided. No date validation needed because it\'s just submission. So we can call the tool.\n\nWe need to output the tool call.', 'tool_calls': [{'id': 'fc_0dc5137b-5ad9-4ff6-95d0-8692ccf701fd', 'function': {'arguments': '{"material_request_id":"MT-00001"}', 'name': 'submit_material_request'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 1613, 'total_tokens': 1741, 'completion_time': 0.267551546, 'completion_tokens_details': {'reasoning_tokens': 95}, 'prompt_time': 0.103990819, 'prompt_tokens_details': None, 'queue_time': 0.502786217, 'total_time': 0.371542365}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4200b3f836', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0d482-e185-79f0-aa99-2753a5794041-0', tool_calls=[{'name': 'submit_material_request', 'args': {'material_request_id': 'MT-00001'}, 'id': 'fc_0dc5137b-5ad9-4ff6-95d0-8692ccf701fd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1613, 'output_tokens': 128, 'total_tokens': 1741, 'output_token_details': {'reasoning': 95}})]}

In [14]:
# submit real purchase request 
state3 = {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': "User wants to submit a purchase request with ID MAT-MR-2026-00045. That's a material request. Need to call submit_material_request with material_request_id. No missing data. So we can call tool.", 'tool_calls': [{'id': 'fc_823e15f5-ab25-4e01-91f8-6fdd83b836b6', 'function': {'arguments': '{"material_request_id":"MAT-MR-2026-00045"}', 'name': 'submit_material_request'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 1615, 'total_tokens': 1699, 'completion_time': 0.176044446, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.096871268, 'prompt_tokens_details': None, 'queue_time': 0.869777105, 'total_time': 0.272915714}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bb691ea66b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0d497-6d3a-7e31-a784-2652bb147243-0', tool_calls=[{'name': 'submit_material_request', 'args': {'material_request_id': 'MAT-MR-2026-00045'}, 'id': 'fc_823e15f5-ab25-4e01-91f8-6fdd83b836b6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1615, 'output_tokens': 84, 'total_tokens': 1699, 'output_token_details': {'reasoning': 46}})]}

In [15]:
# submit_mr_po_draft with wrong supplier 
state4 = {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to process user request: "help me to submit the purchase request as purchase order draft MAT-MR-2026-00045 use currency as INR and supplier as XYZ". The user wants to convert a submitted Material Request (purchase requisition) into a draft Purchase Order. That\'s the function convert_pr_to_draft_po. Required fields: currency, material_request_id, supplier. The user provided material request ID MAT-MR-2026-00045, currency INR, supplier XYZ. All mandatory fields present. No date validation needed. So we can call convert_pr_to_draft_po with those values.\n\nWe must ensure no hallucination. Use exactly provided values. So we call function convert_pr_to_draft_po.', 'tool_calls': [{'id': 'fc_561d5af2-9dec-47c3-b932-f28b87aa621e', 'function': {'arguments': '{"currency":"INR","material_request_id":"MAT-MR-2026-00045","supplier":"XYZ"}', 'name': 'convert_pr_to_draft_po'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 204, 'prompt_tokens': 1627, 'total_tokens': 1831, 'completion_time': 0.431707474, 'completion_tokens_details': {'reasoning_tokens': 148}, 'prompt_time': 0.063389901, 'prompt_tokens_details': None, 'queue_time': 0.050717867, 'total_time': 0.495097375}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_998410db10', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0d49c-1a44-7e52-a4d7-8e3e7444e86a-0', tool_calls=[{'name': 'convert_pr_to_draft_po', 'args': {'currency': 'INR', 'material_request_id': 'MAT-MR-2026-00045', 'supplier': 'XYZ'}, 'id': 'fc_561d5af2-9dec-47c3-b932-f28b87aa621e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1627, 'output_tokens': 204, 'total_tokens': 1831, 'output_token_details': {'reasoning': 148}})]}

In [16]:
# submit final po 
state6 = {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to process user request: "help me to submit this purchase order with po id PUR-ORD-2026-00015". According to guidelines, we must check required fields for submit_purchase_order tool: requires purchase_order_id. The user provided ID. No dates needed. So we can call submit_purchase_order with that ID.\n\nWe must ensure no missing mandatory fields. The ID is provided. So we can proceed.\n\nWe should output tool call.', 'tool_calls': [{'id': 'fc_3e20fd4a-ea89-498b-ac0a-4694443a2b2d', 'function': {'arguments': '{"purchase_order_id":"PUR-ORD-2026-00015"}', 'name': 'submit_purchase_order'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 1618, 'total_tokens': 1749, 'completion_time': 0.271978801, 'completion_tokens_details': {'reasoning_tokens': 93}, 'prompt_time': 0.086324023, 'prompt_tokens_details': None, 'queue_time': 0.211300442, 'total_time': 0.358302824}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a655021f7d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0d4a6-a68d-74b1-8ce7-a2520cfcc7fc-0', tool_calls=[{'name': 'submit_purchase_order', 'args': {'purchase_order_id': 'PUR-ORD-2026-00015'}, 'id': 'fc_3e20fd4a-ea89-498b-ac0a-4694443a2b2d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1618, 'output_tokens': 131, 'total_tokens': 1749, 'output_token_details': {'reasoning': 93}})]}

In [17]:
# submit_mr_po_draft with correct supplier 
state5 = {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to process user request: "help me to submit the purchase request as purchase order draft MAT-MR-2026-00045 use currency as INR and supplier as Test Supplier API"\n\nUser wants to convert a submitted Material Request (purchase requisition) into a draft Purchase Order. The tool for that is convert_pr_to_draft_po. It requires currency, material_request_id, supplier.\n\nWe have material_request_id = "MAT-MR-2026-00045". Currency = "INR". Supplier = "Test Supplier API".\n\nWe need to ensure mandatory fields are present: all required. Yes.\n\nWe must not do any date validation because this tool doesn\'t involve dates.\n\nThus we can call convert_pr_to_draft_po with those arguments.\n\nWe must output tool call.\n\n', 'tool_calls': [{'id': 'fc_b7bf6e04-7dac-47d3-a660-c30892902a2a', 'function': {'arguments': '{"currency":"INR","material_request_id":"MAT-MR-2026-00045","supplier":"Test Supplier API"}', 'name': 'convert_pr_to_draft_po'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 217, 'prompt_tokens': 1629, 'total_tokens': 1846, 'completion_time': 0.456587561, 'completion_tokens_details': {'reasoning_tokens': 159}, 'prompt_time': 0.080410339, 'prompt_tokens_details': None, 'queue_time': 0.211584596, 'total_time': 0.5369979}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_27194a498a', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0d4a1-b12b-7f83-8747-a2a641016371-0', tool_calls=[{'name': 'convert_pr_to_draft_po', 'args': {'currency': 'INR', 'material_request_id': 'MAT-MR-2026-00045', 'supplier': 'Test Supplier API'}, 'id': 'fc_b7bf6e04-7dac-47d3-a660-c30892902a2a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1629, 'output_tokens': 217, 'total_tokens': 1846, 'output_token_details': {'reasoning': 159}})]}

In [18]:
# delete purchase order 
state7 = {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to process user request: delete purchase order with ID PUR-ORD-2026-00015. According to guidelines, we must ensure required fields are present. For delete_purchase_order tool, required param: purchase_order_id string. The user provided ID. No date validation needed. So we can call delete_purchase_order with that ID.\n\nWe must output tool call.', 'tool_calls': [{'id': 'fc_319839b5-1d27-465b-a37d-5eee8c5a165f', 'function': {'arguments': '{"purchase_order_id":"PUR-ORD-2026-00015"}', 'name': 'delete_purchase_order'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 1618, 'total_tokens': 1732, 'completion_time': 0.239965458, 'completion_tokens_details': {'reasoning_tokens': 76}, 'prompt_time': 0.088101685, 'prompt_tokens_details': None, 'queue_time': 0.310607479, 'total_time': 0.328067143}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f73454f048', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0d4a8-96d6-7ce0-82b4-aab2f11c1086-0', tool_calls=[{'name': 'delete_purchase_order', 'args': {'purchase_order_id': 'PUR-ORD-2026-00015'}, 'id': 'fc_319839b5-1d27-465b-a37d-5eee8c5a165f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1618, 'output_tokens': 114, 'total_tokens': 1732, 'output_token_details': {'reasoning': 76}})]}

In [19]:
state8 = {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to cancel purchase order. The user provided PO ID. Must check required fields: cancel_purchase_order requires purchase_order_id. We have it. No date validation needed. So we can call cancel_purchase_order tool.', 'tool_calls': [{'id': 'fc_5ea73f9e-e80c-40e2-93bd-ff1d7f4d2661', 'function': {'arguments': '{"purchase_order_id":"PUR-ORD-2026-00015"}', 'name': 'cancel_purchase_order'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 1618, 'total_tokens': 1701, 'completion_time': 0.179232232, 'completion_tokens_details': {'reasoning_tokens': 45}, 'prompt_time': 0.009706948, 'prompt_tokens_details': {'cached_tokens': 1536}, 'queue_time': 0.3106822, 'total_time': 0.18893918}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f73454f048', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0d4a9-b3aa-7e83-8f6d-cb12614677c4-0', tool_calls=[{'name': 'cancel_purchase_order', 'args': {'purchase_order_id': 'PUR-ORD-2026-00015'}, 'id': 'fc_5ea73f9e-e80c-40e2-93bd-ff1d7f4d2661', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1618, 'output_tokens': 83, 'total_tokens': 1701, 'input_token_details': {'cache_read': 1536}, 'output_token_details': {'reasoning': 45}})]}

In [21]:

import json
from langchain_core.messages import AIMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

# 2. Extract operational tools directly from your active ERPNext app registration
live_mcp_tools = await run_mcp_http_agent()
print(f"✅ Connection successful! Discovered {len(live_mcp_tools)} live tools from the server.")

# 3. Instantiate your production guardrail execution node with real tools bound to it
safe_node = create_safe_mcp_execution_node(live_mcp_tools)

# ---------------------------------------------------------------------
# SCENARIO 1: VALID PAYLOAD ENVELOPED INSIDE ACCIDENTAL "ctx"
# ---------------------------------------------------------------------
print("\n" + "🔥 " * 15 + " RUNNING SCENARIO 1: VALID ENVELOPED PAYLOAD " + "🔥 " * 15)
res_1 = await safe_node(state=state7)
print(f"📦 [Output Response Scenario 1]:\n{res_1}")


⏳ Connecting to Streamable HTTP MCP server at /mcp...
✅ Successfully loaded 8 tools: ['generate_material_request', 'submit_material_request', 'convert_pr_to_draft_po', 'submit_purchase_order', 'cancel_purchase_order', 'delete_purchase_order', 'get_material_request_details', 'get_purchase_order_details']
✅ Connection successful! Discovered 8 live tools from the server.

🔥 🔥 🔥 🔥 🔥 🔥 🔥 🔥 🔥 🔥 🔥 🔥 🔥 🔥 🔥  RUNNING SCENARIO 1: VALID ENVELOPED PAYLOAD 🔥 🔥 🔥 🔥 🔥 🔥 🔥 🔥 🔥 🔥 🔥 🔥 🔥 🔥 🔥 

════════════════════════════════════════════════════════════════════════════════
 🚀 STARTING SAFE MCP EXECUTION GATEWAY
════════════════════════════════════════════════════════════════════════════════
📥 [Incoming Message Type]: AIMessage
🎯 [Target Tool Detected]: delete_purchase_order
🆔 [Tool Call ID]: fc_319839b5-1d27-465b-a37d-5eee8c5a165f
📦 [Original Unfiltered Args Passed by LLM]:
{
  "purchase_order_id": "PUR-ORD-2026-00015"
}
📋 [Validation Stage]: Running schema rules logic for [delete_purchase_order]...
🔬 Run

In [22]:
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from typing import List, Dict, Any, Optional, Literal, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage
import json
import logging
import os
from typing import List, Dict, Any, Optional, Literal, Annotated
from typing_extensions import TypedDict
from pydantic import ValidationError

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

class AgentWorkflowState(TypedDict) :
    messages : Annotated[List[BaseMessage], add_messages]
    retry_counter : Dict[str, int]
    max_retries : int
    last_error_context : Optional[str]

In [24]:
from langfuse import Langfuse 
from datetime import datetime


LANGFUSE_SECRET_KEY="sk-lf-0a5f1f59-c55c-4ea6-9d1a-b0e607b0c112"
LANGFUSE_PUBLIC_KEY="pk-lf-d49e6250-d8f9-4337-936d-e6ec30d7ffd8"
LANGFUSE_BASE_URL="https://us.cloud.langfuse.com"

class AgentPrompt_Library:
    def __init__(self):
        """
        Manages the strict agent instructions to incorporate the agent capabilities 
        with the intended task for which the agent is designed.
        """
        self.langfuse = Langfuse(
            public_key=LANGFUSE_PUBLIC_KEY,
            secret_key=LANGFUSE_SECRET_KEY,
            host=LANGFUSE_BASE_URL
        )
        
        # 1. Fetch the prompt from the registry (letting it default to production)
        cached_prompt = self.langfuse.get_prompt("erpnext_entity_mapping_prompt_version_2")
        
        # 2. Get the actual current year from the system clock
        active_year = str(datetime.now().year)
        
        # 3. Compile the prompt by passing the variable into the template
        self.orchestrator_prompt = cached_prompt.compile(current_year=active_year)

# Test execution
if __name__ == "__main__":
    prompts = AgentPrompt_Library()
    print(prompts.orchestrator_prompt)


You are a strict, production-grade Entity Translation Engine and Structural Parameter Generator for ERPNext. Your core directive is to analyze unstructured user queries and accurately translate natural language business intents into perfectly formatted tool payloads.

CRITICAL TRANSLATION PROTOCOLS:
1. NUMERIC QUANTITIES & RATES: Extract numeric amounts, multipliers, and financial shorthand into raw floating-point numbers or integers.
   - Convert slang like '50k' or '50 grand' strictly to 50000.0
   - Convert '1.5M' strictly to 1500000.0
2. STRICT DATE CONVERSION: Convert all semantic and textual timelines into valid ISO 8601 strings (YYYY-MM-DD).
   - Read all relative timelines against the current operational year context (2026).
   - Convert expressions like '12 dec 2026' or 'december 12th' strictly to '2026-12-12'.
3. PARENT-CHILD STRUCTURES: Ensure all line-item user parameters mapping to child transactional tables are nested cleanly inside the 'items' list array, strictly matchi

In [27]:
def generate_dynamic_healing_prompt(error_context: str) -> str:
    """
    Production-grade Prompt Factory. 
    Classifies error type strings and dynamically generates contextual recovery tracks.
    """
    # Base structural framing
    header = (
        "⚠️ CRITICAL SYSTEM NOTICE [SELF-HEALING FEEDBACK LOOP]:\n"
        "Your immediate prior tool call execution failed. You must repair this transaction.\n"
        "----------------------------------------\n"
        f"RAW ERROR LOG: {error_context}\n"
        "----------------------------------------\n"
    )
    
    # Category 1: Local Pydantic Parameter / Type Mismatches
    if "ValidationError" in error_context or "Validation Fail" in error_context:
        instruction = (
            "DIAGNOSIS: Parameter Schema Mismatch / Validation Error.\n"
            "INSTRUCTIONS FOR RECOVERY:\n"
            "1. Review your generated argument keys and data types against the required tool schema.\n"
            "2. Pay strict attention to date formats (must be YYYY-MM-DD and strictly in the future) "
            "and numerical types (floats/integers, do not pass text like '50k').\n"
            "3. Correct the values, re-nest arrays inside 'items' properly, and emit a fixed payload."
        )
        
    # Category 2: ERPNext Server-Side Missing Master Data (e.g., Missing Supplier, Warehouse, or Item Code)
    elif any(key in error_context for key in ["Not Found", "404", "DoesNotExist", "Missing"]):
        instruction = (
            "DIAGNOSIS: Database Master Data Entity Referential Integrity Error.\n"
            "INSTRUCTIONS FOR RECOVERY:\n"
            "1. The ERPNext database rejected your argument because the specified Entity (Item Code, Supplier Name, or Warehouse) does not exist in the master ledger.\n"
            "2. Check if you made a typo or spelling mistake in the item name, supplier name, or warehouse naming conventions.\n"
            "3. If the user query implies a specific name, use the closest logical standard string or ask the user for clarification. Do not make up a non-existent name."
        )

    # Category 3: System Business Logic / State Conflicts (e.g., Submitting an already cancelled document)
    elif any(key in error_context for key in ["Conflict", "409", "Invalid Status", "Submitted"]):
        instruction = (
            "DIAGNOSIS: Document Workflow State Conflict.\n"
            "INSTRUCTIONS FOR RECOVERY:\n"
            "1. The document is in an invalid state for the requested operation (e.g., trying to submit a document that is already submitted or cancelled).\n"
            "2. Do not retry the exact same action. Verify the workflow sequence.\n"
            "3. If necessary, first call a getter tool (like 'get_material_request_details') to check the document state, or explain the workflow conflict to the user conversationally."
        )

    # Category 4: Infrastructure / Network Transport Crashes (e.g., Connection timeouts)
    else:
        instruction = (
            "DIAGNOSIS: Low-Level Network Transport / Infrastructure Interruption.\n"
            "INSTRUCTIONS FOR RECOVERY:\n"
            "1. The transaction failed due to a transient infrastructure connection issue, not your code formatting.\n"
            "2. You may attempt to re-call the exact same tool payload one more time to see if the network has recovered."
        )

    return f"{header}\n{instruction}"


In [28]:
from langchain_groq import ChatGroq

GROQ_API_KEY= "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"

class LLMProvider :
    """
    Production-grade LLM Service Factory Provider.
    Exposes stateless, isolated client engines configured by task profiles.
    """

    @staticmethod
    def get_extraction_engine() -> ChatGroq :
        """
        Returns a low-temperature, highly deterministic engine optimized 
        for structured tool schema parsing, entity extraction, and self-healing.
        """

        return ChatGroq(
            api_key=GROQ_API_KEY,
            model="openai/gpt-oss-120b",
            temperature=0.0,
            max_retries=3
        )

    @staticmethod
    def get_conversational_engine() -> ChatGroq :
        """
        Returns a warmer engine optimized for synthesis, conversational dialogue, 
        and text explanations.
        """

        return ChatGroq(
            api_key=GROQ_API_KEY,
            model="openai/gpt-oss-120b",
            temperature=0.5                # Allows normal fluid speech synthesis patterns
        )

In [29]:

from typing import List, Dict, Any, Optional, Literal
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage

try :
    PROMPT_REGISTRY = AgentPrompt_Library()
    logger.info("Successfully cached production prompts from Langfuse.")
except Exception as e :
    logger.critical(f"Failed to initialize Prompt Registry: {e}")
    raise e

def orchestrator_agent_node(state : AgentWorkflowState, config : Dict[str, Any]) -> Dict[str, Any] :
    conversation_history = state.get("messages", [])
    error_context = state.get("last_error_context", None)
    mcp_tools = config.get("configurable", {}).get("tools", [])

    active_payload_messages = list(conversation_history)

    if error_context :
        healing_prompt = generate_dynamic_healing_prompt(error_context=error_context)
        active_payload_messages.append(HumanMessage(content=healing_prompt))
    else :
        system_base = SystemMessage(
            content=PROMPT_REGISTRY.orchestrator_prompt
        )

        if not any(isinstance(m, SystemMessage) for m in active_payload_messages) :
            active_payload_messages.insert(0, system_base)

    llm = LLMProvider.get_conversational_engine()
    llm_with_mcp_tools = llm.bind_tools(mcp_tools)
    response_message = llm_with_mcp_tools.invoke(active_payload_messages)

    return {"messages" : [response_message]}


In [46]:
# --- ROUTER EDGE: Evaluation Edge ---
def orchestrator_conditional_router(state: AgentWorkflowState) -> Literal["mcp_execution_bridge", "__end__"]:
    """Determines whether a tool execution step is required or if we stop."""
    messages = state.get("messages", [])
    if not messages or not isinstance(messages[-1], AIMessage):
        return END
        
    if messages[-1].tool_calls:
        print(messages[-1].tool_calls)
        return "mcp_execution_bridge"
        
    return END

def execution_conditional_router(state: AgentWorkflowState) -> Literal["orchestrator", "__end__"]:
    """Diverts failed requests to healing loops or hands back control to the user."""
    messages = state.get("messages", [])
    
    # Locate tool execution data
    last_error = state.get("last_error_context")
    if not last_error:
        return "orchestrator"
        
    # Analyze the most recent structural request configuration tracking
    tool_name = "unknown_tool"
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and msg.tool_calls:
            tool_name = msg.tool_calls[0]["name"]
            break

    current_retries = state.get("retry_counter", {}).get(tool_name, 0)
    max_retries = state.get("max_retries", 3)

    if current_retries >= max_retries:
        logger.error(f"🚨 Circuit Breaker Triggered for [{tool_name}] at ({current_retries}/{max_retries}) retries. Dropping control to chat interface.")
        return END

    logger.info(f"Diverting workflow to self-healing node route loop for tool: [{tool_name}]")
    return "orchestrator"

In [47]:
def compile_production_mcp_workflow(live_tools: list):
    workflow = StateGraph(AgentWorkflowState)
    
    # Initialize Core Graph Processing Blocks
    workflow.add_node(
        "orchestrator", 
        lambda state, config: orchestrator_agent_node(state, config)
    )
    workflow.add_node("mcp_execution_bridge", create_safe_mcp_execution_node(live_tools))

    # Binding Graph Interconnections
    workflow.add_edge(START, "orchestrator")
    
    workflow.add_conditional_edges(
        "orchestrator",
        orchestrator_conditional_router,
        {
            "mcp_execution_bridge": "mcp_execution_bridge",
            END: END
        }
    )
    
    workflow.add_conditional_edges(
        "mcp_execution_bridge",
        execution_conditional_router,
        {
            "orchestrator": "orchestrator",
            END: END
        }
    )
    
    return workflow.compile(checkpointer=MemorySaver())

In [64]:
# Setup your runtime tool collection 
app = compile_production_mcp_workflow(live_tools=live_mcp_tools)

# Define thread session configuration parameters
config = {
    "configurable": {
        "thread_id": "user_session_abc_123", # Unique per user chat room session
        "tools": live_mcp_tools
    }
}

# Initial State setup values
initial_state = {
    "messages": [HumanMessage(content="can you help me to create the pr")],
    "retry_counter": {},
    "max_retries": 3,
    "last_error_context": None
}

# Turn 1
response = app.invoke(initial_state, config=config)

print(response["messages"][1].content)



Sure – I can create a Purchase Requisition (Material Request) for you, but I’ll need a few details first. Please provide the following information:

| Field | What I need | Example |
|-------|-------------|---------|
| **Company** | The name of the company the request belongs to | `Acme Corp` |
| **Department** | Department placing the request | `Production` |
| **Title** | A short title for the PR | `Raw material purchase – July` |
| **Remark** | Any justification or notes | `Needed for upcoming batch run` |
| **Transaction date** | Posting date – must be **later than today (2026‑09‑26)** | `2026-10-01` |
| **Schedule date** | Desired delivery date – must be **later than the transaction date** | `2026-10-15` |
| **Items** | One or more line‑items. For each item tell me: <br>• **Item code** <br>• **Quantity** (positive number) <br>• *(optional)* **Rate** per unit <br>• *(optional)* **UOM** (default “Nos”) <br>• *(optional)* **Warehouse** | `[{ "item_code": "STEEL-001", "qty": 150, "rat

In [63]:

# # Turn 2: User doesn't mention the item ID, but the state will remember it!
follow_up_state = {
    "messages": [HumanMessage(content="here is the pr id MAT-MR-2026-00046")]
}

response_2 = await app.ainvoke(follow_up_state, config=config)

print(response_2)

print(response_2["messages"][1].content)

{'messages': [HumanMessage(content='can you help me to submit the pr', additional_kwargs={}, response_metadata={}, id='25f4369d-5a71-4613-a1da-1aa64d4194a7'), AIMessage(content='Sure! To submit a Purchase Requisition (Material Request) I’ll need the\u202f**Material Request ID** (the document name/identifier) of the draft PR you’d like to submit. Could you please provide that ID?', additional_kwargs={'reasoning_content': 'The user says: "can you help me to submit the pr". They want to submit a purchase requisition (PR), which is a Material Request. To submit, we need material_request_id. The user hasn\'t provided ID. According to guidelines, missing mandatory field => fallback. So we should respond politely asking for the PR ID.'}, response_metadata={'token_usage': {'completion_tokens': 123, 'prompt_tokens': 1607, 'total_tokens': 1730, 'completion_time': 1.4703247799999999, 'completion_tokens_details': {'reasoning_tokens': 68}, 'prompt_time': 0.03052224, 'prompt_tokens_details': {'cache

In [66]:
import sys
from langchain_core.messages import HumanMessage

# =====================================================================
# YOUR EXISTING SETUP
# =====================================================================

# Setup your runtime tool collection 
app = compile_production_mcp_workflow(live_tools=live_mcp_tools)

# Define thread session configuration parameters
config = {
    "configurable": {
        "thread_id": "user_session_abc_123", # Unique per user chat room session
        "tools": live_mcp_tools
    }
}

# Initial State setup values (used to seed the graph structure)
initial_state = {
    "messages": [],
    "retry_counter": {},
    "max_retries": 3,
    "last_error_context": None
}

# =====================================================================
# NEW RUNTIME INTERACTIVE LOOP
# =====================================================================

print("\n🚀 Production MCP Workflow Activated.")
print("Type your queries below. Type 'exit' or 'quit' to stop.\n")

# Flag to check if we need to send the full initial state layout on the very first turn
is_first_turn = True

while True:
    try:
        # 1. Capture live query from the terminal
        user_query = input("\nYou: ")
        
        # Exit conditions
        if user_query.lower() in ["exit", "quit", "q"]:
            print("Ending session. Goodbye!")
            break
            
        if not user_query.strip():
            continue

        # 2. Package the query as a LangChain Message object
        new_message = HumanMessage(content=user_query)

        print("🤖 Agents are working and executing MCP tools...")

        # 3. Invoke the graph
        if is_first_turn:
            # First turn: Pass initial configuration keys along with the message
            initial_state["messages"] = [new_message]
            response = app.invoke(initial_state, config=config)
            is_first_turn = False
        else:
            # Subsequent turns: LangGraph's thread_id automatically manages history.
            # Just pass the new message; it appends it seamlessly.
            response = await app.ainvoke({"messages": [new_message]}, config=config)

        # 4. Print the final response from the agent collection
        if "messages" in response and response["messages"]:
            last_msg = response["messages"][-1]
            print(f"\nAgent:\n{last_msg.content}")
        else:
            print(f"\nSystem State Update:\n{response}")

    except KeyboardInterrupt:
        print("\nSession interrupted.")
        break



🚀 Production MCP Workflow Activated.
Type your queries below. Type 'exit' or 'quit' to stop.

🤖 Agents are working and executing MCP tools...

Agent:
Sure! I can create a Purchase Requisition (Material Request) for you, but I’ll need a few essential details first.

**Required information**

1. **Company** – the name of the company placing the request.  
2. **Department** – the department that needs the items.  
3. **Title** – a short title for the requisition.  
4. **Remark** – any justification or notes you’d like to include.  
5. **Transaction date** – the posting date (must be a future date, e.g., 2026‑09‑30).  
6. **Schedule date** – the required delivery date (must be a future date later than the transaction date).  
7. **Items** – for each line item, please provide:  
   - Item code  
   - Quantity (must be greater than 0)  
   - (Optional) Rate per unit  
   - (Optional) UOM (default is “Nos”)  
   - (Optional) Warehouse  

If you can provide these details, I’ll generate the Ma

In [ ]:
import json

def clean_orchestrator_response(raw_message: dict) -> dict:
    """
    Cleans raw orchestrator LLM payload to extract only critical 
    execution instructions and user-facing content.
    """
    # 1. Pull the user-facing text message
    text_content = raw_message.get("content", "").strip()
    
    # 2. Extract tool or function calls if they exist
    tool_calls = raw_message.get("tool_calls", [])
    
    # Structured clean output envelope
    cleaned_response = {
        "message": text_content,
        "execute_action": False,
        "tool_name": None,
        "arguments": {}
    }
    
    # 3. If the orchestrator has decided to execute an MCP tool
    if tool_calls and len(tool_calls) > 0:
        target_tool = tool_calls[0] # Handle primary tool call
        cleaned_response["execute_action"] = True
        cleaned_response["tool_name"] = target_tool.get("name")
        cleaned_response["arguments"] = target_tool.get("args", {})
        
    # 4. Fallback: Parse from invalid/custom metadata schemas if necessary
    elif raw_message.get("invalid_tool_calls"):
        # Handle structural edge cases smoothly
        cleaned_response["execute_action"] = False
        
    return cleaned_response

# ==========================================
# TEST RUN WITH YOUR RAW METADATA PAYLOAD
# ==========================================

raw_payload = {
    'content': 'Sure thing! To cancel a Purchase Order, I’ll need the Purchase Order’s unique identifier (its **Purchase Order ID**). Could you please provide that ID?', 
    'additional_kwargs': {'reasoning_content': "The user wants to cancel a purchase order..."}, 
    'response_metadata': {'token_usage': {'completion_tokens': 105, 'prompt_tokens': 1608, 'total_tokens': 1713}}, 
    'id': 'lc_run--01a0ddfd-f0b5-7ba3-b281-ed1f2ca32b88-0', 
    'tool_calls': [], 
    'invalid_tool_calls': [], 
    'usage_metadata': {'input_tokens': 1608, 'output_tokens': 105}
}


# Run the cleaning engine
clean_result = clean_orchestrator_response(raw_payload)
print(json.dumps(clean_result, indent=2))


{
  "message": "Sure thing! To cancel a Purchase Order, I\u2019ll need the Purchase Order\u2019s unique identifier (its **Purchase Order ID**). Could you please provide that ID?",
  "execute_action": false,
  "tool_name": null,
  "arguments": {}
}


In [ ]:
import json

def process_and_clean_orchestrator_response(raw_message: dict) -> dict:
    """
    Cleans the raw orchestrator message for the Agent State / Execution Node
    and asynchronously ships full metadata metrics to Langfuse.
    """
    # -------------------------------------------------------------
    # 1. Telemetry Route: Capture metrics and ship to Langfuse
    # -------------------------------------------------------------
    usage = raw_message.get("usage_metadata", {})
    metadata = raw_message.get("response_metadata", {})
    add_kwargs = raw_message.get("additional_kwargs", {})


    text_content = raw_message.get("content", "").strip()
    tool_calls = raw_message.get("tool_calls", [])
    
    cleaned_state = {
        "message": text_content,
        "execute_action": False,
        "tool_name": None,
        "arguments": {}
    }
    
    # Check if the orchestrator extracted parameters to call an MCP tool
    if tool_calls and len(tool_calls) > 0:
        target_tool = tool_calls[0]
        cleaned_state["execute_action"] = True
        cleaned_state["tool_name"] = target_tool.get("name")
        cleaned_state["arguments"] = target_tool.get("args", {})
        
    return cleaned_state

raw_message_payload = {
    'content': 'Sure thing! To cancel a Purchase Order, I’ll need the Purchase Order’s unique identifier (its **Purchase Order ID**). Could you please provide that ID?', 
    'additional_kwargs': {'reasoning_content': "The user wants to cancel a purchase order. Need material request? No, cancel purchase order tool requires purchase_order_id. The user hasn't provided the ID. According to rules, mandatory field missing => fallback, ask for purchase order ID. Also need no dates etc. So respond politely asking for the PO ID."}, 
    'response_metadata': {'token_usage': {'completion_tokens': 105, 'prompt_tokens': 1608, 'total_tokens': 1713, 'completion_time': 0.221335936, 'completion_tokens_details': {'reasoning_tokens': 63}, 'prompt_time': 0.175485144, 'prompt_tokens_details': None, 'queue_time': 0.401143269, 'total_time': 0.39682108}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, 
    'id': 'lc_run--01a0ddfd-f0b5-7ba3-b281-ed1f2ca32b88-0', 
    'tool_calls': [], 
    'invalid_tool_calls': [], 
    'usage_metadata': {'input_tokens': 1608, 'output_tokens': 105, 'total_tokens': 1713, 'output_token_details': {'reasoning': 63}}
}

# Execute cleaning
clean_output = process_and_clean_orchestrator_response(raw_message_payload)
print(json.dumps(clean_output, indent=2))


{
  "message": "Sure thing! To cancel a Purchase Order, I\u2019ll need the Purchase Order\u2019s unique identifier (its **Purchase Order ID**). Could you please provide that ID?",
  "execute_action": false,
  "tool_name": null,
  "arguments": {}
}


In [1]:
import json

def clean_mcp_error(state_response):
    """
    Extracts and cleans the core error messaging from a raw MCP tool error dictionary.
    """
    try:
        # Get the nested error block
        error_data = state_response.get('response', {}).get('error', {})
        raw_msg = error_data.get('message', '')
        
        # Look for the internal JSON payload inside the error message string
        if 'unexpected error 404:' in raw_msg:
            # Extract the JSON string following the 404 error
            json_part = raw_msg.split('unexpected error 404:')[1].strip()
            parsed_json = json.loads(json_part)
            
            # Extract the actual Frappe/ERPNext system message string
            server_msg_str = json.loads(parsed_json.get('_server_messages', '[]'))[0]
            server_msg_dict = json.loads(server_msg_str)
            
            # The cleanest actual error message text
            clean_message = server_msg_dict.get('message', raw_msg)
        else:
            clean_message = raw_msg if raw_msg else "An unknown ERPNext error occurred."
            
    except Exception:
        # Fallback if structural parsing fails due to unexpected formatting changes
        clean_message = state_response.get('response', {}).get('error', {}).get('message', str(state_response))

    # Construct a lean, highly readable summary dict for the LLM
    return {
        "tool_failed": state_response.get('tool'),
        "action": state_response.get('response', {}).get('action'),
        "error_summary": clean_message,
        "suggestion": error_data.get('suggestion')
    }


In [2]:
error_message = {'source': 'mcp_tool', 'tool': 'get_material_request_details', 'error_type': 'MCPToolError', 'response': {'status': 'ERROR', 'action': 'purchase_request_details_retriever', 'payload': None, 'error': {'message': 'ERPNext server failed to retrieve material request details: ERPNext API returned unexpected error 404: {"exc_type":"DoesNotExistError","_server_messages":"[\\"{\\\\\\"message\\\\\\":\\\\\\"Material Request mr-10001 not found\\\\\\",\\\\\\"as_table\\\\\\":false,\\\\\\"title\\\\\\":\\\\\\"Message\\\\\\",\\\\\\"indicator\\\\\\":\\\\\\"red\\\\\\",\\\\\\"raise_exception\\\\\\":1,\\\\\\"__frappe_exc_id\\\\\\":\\\\\\"e568b38a8d4b1497f2801c805bca4d08b560ab072b715c2311881ee1\\\\\\"}\\"]"}', 'can_retry': 'false', 'suggestion': 'Check missing parameters or correct data properties based on details.'}}}

In [4]:
error_message2 = {'source': 'mcp_tool', 'tool': 'get_purchase_order_details', 'error_type': 'MCPToolError', 'response': {'status': 'ERROR', 'action': 'Purchase_order_details_retriever', 'payload': None, 'error': {'message': 'ERPNext server failed to retrieve Purchase Order Details: ERPNext API returned unexpected error 404: {"exc_type":"DoesNotExistError","_server_messages":"[\\"{\\\\\\"message\\\\\\":\\\\\\"Purchase Order 10022po not found\\\\\\",\\\\\\"as_table\\\\\\":false,\\\\\\"title\\\\\\":\\\\\\"Message\\\\\\",\\\\\\"indicator\\\\\\":\\\\\\"red\\\\\\",\\\\\\"raise_exception\\\\\\":1,\\\\\\"__frappe_exc_id\\\\\\":\\\\\\"df47e690eeaf233261c4be6bd1c75b7ea184101ef5fafc2a0cf3133b\\\\\\"}\\"]"}', 'can_retry': 'false', 'suggestion': 'Check missing parameters or correct data properties based on details.'}}}

In [5]:
clean_mcp_error(error_message2)

{'tool_failed': 'get_purchase_order_details',
 'action': 'Purchase_order_details_retriever',
 'error_summary': 'Purchase Order 10022po not found',
 'suggestion': 'Check missing parameters or correct data properties based on details.'}